####1. Create a dataframe as below
```
  +---+--------+---+------+
  | id|    name|age|salary|
  +---+--------+---+------+
  |100|Prashant| 45| 45000|
  |101|   Tarun| 36| 33000|
  |102|   David| 48| 28000|
  +---+--------+---+------+
```

In [0]:
schema = """
    id int,
    name string,
    age  short,
    salary double
"""
data_list = [(100, "Prashant", 45, 45000),
             (101, "Tarun", 36, 33000),
             (102, "David", 48, 28000)]

#sample_df = spark.createDataFrame(data=data_list)
#sample_df = spark.createDataFrame(data=data_list).toDF("id", "name", "age", "salary") # It allows to remane all the colums in this order if you're not giving any schema or Dtypes(Not Recommended)
sample_df = spark.createDataFrame(data=data_list, schema=schema)
display(sample_df)

####2. Add following columns to your Dataframe
1. increment: 10% of the salary up to 3000 maximum increment
2. revised_salary: salary + increment

In [0]:
from pyspark.sql.functions import expr

salary_df= (
    sample_df.withColumns(
        {
            "increment": expr("CASE when salary<30000 THEN salary * 0.1 ELSE 3000 END"),
            "RevisedSalary": expr("salary + increment") # This is allowed only when you're creating a new column
            #"salary": expr("salary + increment") # This is not allowed as you're updating an existing column
        }
    )

)

display(salary_df)

####3. Add following columns to your Dataframe
* increment: 10% of the salary up to 3000 maximum increment

Replace the following column in your dataframe
* salary: current salary + increment

In [0]:
salary_df2= (
    sample_df.withColumns(
        {
            "increment": expr("CASE when salary<30000 THEN salary * 0.1 ELSE 3000 END"),
            "salary": expr("salary + increment") # This is not allowed as you're updating an existing column
        }
    )
)



In [0]:
#2 ways to resolve above issue
#1. Rather than adding using inc

salary_df3= (
    sample_df.withColumns(
        {
            "increment": expr("CASE when salary<30000 THEN salary * 0.1 ELSE 3000 END"),
            "salary": expr("salary + CASE when salary<30000 THEN salary * 0.1 ELSE 3000 END") # This is not allowed as you're updating an existing column
        }
    )
)

salary_df3.show()

In [0]:
#2. Using 2 with column(Basically we're breaking the transformation 1st we're calculating the increment and then refering the increment this way it will work) #Avoid using this way it if possible
from pyspark.sql.functions import expr

salary_df4 = (
    sample_df.withColumn("increment", expr("case when salary > 30000 then 3000 else salary * 10/100 end"))
        .withColumn("salary", expr("salary + increment"))
)

salary_df4.display() 

####4. Add a batch number (uuid) column to your dataframe

In [0]:
import uuid # This is used to generate a unique id Universal Unique Identifier
from pyspark.sql.functions import lit  #lit is used to create a literal value and helps to add a new column in dataframe

batch_id = str(uuid.uuid4())

salary_batch_df = salary_df.withColumn("batch_id", lit(batch_id))

salary_batch_df.display()

####4. Rename the dataframe colums as listed below
1. increment - annual_increment
2. salary - incremented_salary

In [0]:
new_salary_df = (salary_df3.withColumnsRenamed(
    {"salary" : "incremented_salary",
     "increment": "annual_increment"  
    }
))

new_salary_df.display()

####5. Remove the following colums from your dataframe
1. age
2. annual_increment

In [0]:
small_salary_df = new_salary_df.drop("age","annual_increment")

sample_df.display()